# Topic Identification and Classification of Short Texts
### IT5072 – Natural Language Processing · Coursework 02 · Student ID: MS26900636

**Goal.** Build a model that reads any **three-sentence text** and assigns it to one of **five classes**:
**Medical**, **Sports**, **Religion**, **Politics**, or **Other** (a text that belongs to none of the four given domains).

**Approach in one line.** Unsupervised topic modeling (LDA and NMF) is used to *discover and explain* the latent themes of the
corpora, and a supervised TF-IDF classifier with a calibrated confidence threshold is used to *decide* the final class,
including the open-set *Other* class.

| Section | Content | Marking criterion |
|---|---|---|
| 1 | Theoretical approach | Theory (30%) |
| 2 – 4 | Setup, data exploration, preprocessing | Implementation (30%) |
| 5 | Evaluation design (splits and 3-sentence chunks) | Implementation |
| 6 | Topic modeling: LDA and NMF | Implementation |
| 7 | Supervised classifiers | Implementation |
| 8 | The *Other* class: confidence threshold | Implementation / Testing |
| 9 | Final evaluation on held-out and unseen-topic data | Testing (30%) |
| 10 | Testing with hand-written outside texts | Testing (30%) |
| 11 | Discussion and conclusion | – |

**Data layout** (all paths are relative to this notebook):
```
data/
├── train/                       # given corpora, one folder per domain
│   ├── medical/sci.med/
│   ├── sports/rec.sport.baseball/, rec.sport.hockey/
│   ├── religion/soc.religion.christian/, talk.religion.misc/
│   └── politics/talk.politics.guns/, talk.politics.mideast/, talk.politics.misc/
└── outside/                     # data NOT taken from the given corpora
    ├── other_20ng/tune/         # unrelated newsgroups used to tune the Other threshold
    ├── other_20ng/test/         # different unrelated newsgroups, used only for testing
    └── handmade/<class>/        # 30 newly written 3-sentence texts (6 per class)
```

## 1. Theoretical Approach

### 1.1 Problem definition
Topic identification is the task of finding the theme of a text. Topic *modeling* answers the question
"what themes exist in this collection?" without labels, while topic *classification* answers
"which known theme does this new text belong to?". This coursework requires both:

* The corpora are already grouped by domain, so the four domains are **known classes** (a supervised, closed-set problem).
* A fifth class, **Other**, has **no training data** at all. Any text from an unseen subject (cooking, finance, technology …)
  must be recognised as "none of the above". This is an **open-set recognition** problem: the model must know when it does not know.
* Inputs are only **three sentences** long (roughly 30–70 words). Short texts are sparse: each contains few of the
  vocabulary words, so the model must rely on a small number of strong, topic-bearing keywords.

The proposed solution is a **hybrid** pipeline:

```
 raw posts ──► cleaning ──► tokens ──┬──► Bag-of-Words ──► LDA ─────────┐  (unsupervised: discover & explain topics)
                                     │                                   ├──► topic → domain mapping, topic-based classifier
                                     ├──► TF-IDF ────────► NMF ─────────┘
                                     │
                                     └──► TF-IDF (1–2-grams) ──► NB / LogReg / SVM ──► class probabilities
                                                                                              │
                                                       max probability < τ ?  ──yes──► "Other"
                                                                              └─no───► Medical / Sports / Religion / Politics
```

### 1.2 Corpus
The given corpora are eight newsgroups from the *20 Newsgroups* collection (Lang, 1995), mapped to four domains:

| Domain | Newsgroups | Posts |
|---|---|---|
| Medical | sci.med | 990 |
| Sports | rec.sport.baseball, rec.sport.hockey | 1,993 |
| Religion | soc.religion.christian, talk.religion.misc | 1,625 |
| Politics | talk.politics.guns, talk.politics.mideast, talk.politics.misc | 2,625 |

The classes are **imbalanced** (Politics has about 2.7× as many posts as Medical), which influences the choice of
evaluation metric (macro-F1) and the use of class weighting.

### 1.3 Preprocessing
Raw Usenet posts contain much text that is either noise or *leakage* (information that reveals the label but would never
appear in a real input):

| Step | Reason |
|---|---|
| Remove headers (`From`, `Organization`, `Lines`, …), keep only `Subject` | Headers name the newsgroup or the author's institution, which would let the model "cheat". The subject line is genuine topical content. |
| Remove quoted replies (`>` lines) and "*X writes:*" lines | Quoted text belongs to another post and duplicates content. |
| Remove signatures (after `--`), e-mail addresses, URLs, ASCII art | Not topical; signatures repeat across posts of the same author and cause over-fitting. |
| Lower-casing, keep alphabetic tokens of length ≥ 3 | Normalises surface forms and removes numbers and punctuation. |
| Stop-word removal (NLTK list + Usenet words like *writes*, *article*) | Function words carry no topic information. |
| Lemmatisation (WordNet) | Groups inflected forms (*doctors → doctor*, *games → game*) and reduces sparsity. Lemmatisation is chosen over stemming because it keeps real words, which makes topics easier to read. |

### 1.4 Text representation (vector space model)
Each document is represented as a vector over the vocabulary $V$.

* **Bag-of-Words (BoW)**: the count $tf(t,d)$ of each term $t$ in document $d$. Word order is ignored. LDA is defined over counts,
  so it uses BoW.
* **TF-IDF** (Salton & Buckley, 1988) re-weights counts so that words frequent in a document but rare in the collection get high weight:

$$\text{tfidf}(t,d) = \big(1 + \log tf(t,d)\big)\cdot \log\frac{1+N}{1+df(t)} + 1$$

  where $N$ is the number of documents and $df(t)$ the number of documents containing $t$. The *sublinear* $1+\log tf$ term stops
  long posts from dominating. Vectors are L2-normalised so that short (3-sentence) and long texts are comparable.
* **n-grams**: unigrams and bigrams are used for classification so that phrases such as *gun control*, *blood pressure* or *world series*
  become features.

### 1.5 Unsupervised topic modeling

**Latent Dirichlet Allocation (LDA)** (Blei, Ng & Jordan, 2003) is a generative probabilistic model. It assumes that each document
is a *mixture of topics* and each topic is a *mixture of words*:

1. For each topic $k = 1..K$: draw a word distribution $\beta_k \sim \text{Dirichlet}(\eta)$.
2. For each document $d$: draw topic proportions $\theta_d \sim \text{Dirichlet}(\alpha)$.
3. For each word position $n$ in $d$: draw a topic $z_{dn} \sim \text{Multinomial}(\theta_d)$, then the word $w_{dn} \sim \text{Multinomial}(\beta_{z_{dn}})$.

Only the words $w$ are observed. Learning reverses this process, estimating the hidden $\theta$ and $\beta$ from the observed words by
maximising $p(w \mid \alpha,\eta)$. The exact posterior is intractable, so gensim uses **online variational Bayes** (Hoffman et al., 2010).
Small Dirichlet parameters encode the belief that a document uses only a few topics and a topic uses only a few words. Here $\alpha$ and $\eta$
are learned from the data (`'auto'`).

**Choosing the number of topics $K$.** $K$ is a hyper-parameter. It is selected with **topic coherence $C_v$** (Röder et al., 2015). $C_v$
measures how often the top words of a topic co-occur in the corpus (using sliding windows, NPMI and cosine similarity). A higher value
means more human-interpretable topics.

**Non-negative Matrix Factorisation (NMF)** (Lee & Seung, 1999) is a linear-algebra alternative. The TF-IDF matrix
$V \in \mathbb{R}^{n\times m}_{\ge 0}$ is approximated by two smaller non-negative matrices:

$$V \approx W H, \qquad \min_{W,H \ge 0} \lVert V - WH\rVert_F^2$$

$W$ (documents × topics) gives each document's topic weights, and $H$ (topics × words) gives each topic's word weights. The non-negativity
makes the factorisation *additive* (parts-based), so the topics are interpretable. NMF is deterministic given its initialisation and often
gives sharper topics on short texts, while LDA gives a full probabilistic model.

**From topics to domains.** Topic models produce *unlabelled* topics. Each topic is mapped to the domain in which it has the highest
average share, $\text{domain}(k) = \arg\max_c \frac{1}{|D_c|}\sum_{d\in D_c}\theta_{dk}$. A new text is then classified by summing
its topic proportions per domain. This gives a fully **topic-based classifier** that can be compared with the supervised one.

### 1.6 Supervised classification
Because labelled data exists, supervised learners can use it directly. Three linear classifiers on TF-IDF features are compared:

* **Multinomial Naive Bayes**: applies Bayes' rule with the "naive" assumption that words are conditionally independent given the class:
  $\hat{c} = \arg\max_c \; \log P(c) + \sum_{t} x_t \log P(t \mid c)$, where $P(t \mid c)$ uses Laplace/Lidstone smoothing. It is fast and a strong baseline for text.
  The class prior $P(c)$ is set to **uniform**. With the empirical prior, the largest class (Politics) would win every uncertain
  text, including off-topic ones that should be rejected as *Other*.
* **Logistic Regression (multinomial / softmax)**: a discriminative model,
  $P(c \mid x) = \dfrac{\exp(w_c^\top x + b_c)}{\sum_j \exp(w_j^\top x + b_j)}$, trained by minimising regularised cross-entropy.
  It outputs well-calibrated probabilities, which matter for the *Other* decision.
* **Linear Support Vector Machine**: finds the maximum-margin hyperplane (hinge loss + L2 penalty). It is often the most accurate model
  on high-dimensional sparse text. SVMs give scores rather than probabilities, so **Platt scaling** (sigmoid calibration, Platt, 1999) is applied with
  `CalibratedClassifierCV`.

`class_weight='balanced'` is used where supported to counter the class imbalance.

A fourth, **hybrid** model, *LDA topic proportions → Logistic Regression*, tests whether the compact topic representation
($K$ features instead of ~100k) is enough for classification.

### 1.7 Detecting the *Other* class (open-set recognition)
No training examples exist for *Other*, and "everything else" cannot be sampled completely. The model therefore uses a **reject option**
based on its confidence (the maximum-softmax-probability baseline of Hendrycks & Gimpel, 2017):

$$\hat{y}(x) = \begin{cases} \arg\max_c P(c\mid x) & \text{if } \max_c P(c\mid x) \ge \tau \\ \text{Other} & \text{otherwise}\end{cases}$$

A text about cooking contains few Medical/Sports/Religion/Politics keywords, so the probability mass is spread out and the maximum is low.
The threshold $\tau$ is **tuned** on a validation set that contains in-domain texts and *Other* texts from three unrelated newsgroups
(computer graphics, X-Windows, cars). It is then **tested** on four *different* unrelated newsgroups (space, for-sale, motorcycles,
electronics) and on hand-written texts. This shows the rule generalises to subjects it never saw, instead of memorising one kind of "other".

### 1.8 Evaluation protocol
* The given posts are split **at document level** into train (70%), validation (15%) and test (15%), stratified by domain.
  Splitting before chunking guarantees that no sentence from a test post was seen in training.
* Models are **trained on whole posts** (more context gives better word statistics) but **evaluated on three-sentence chunks** cut
  from the body of validation/test posts. This matches the real task. The subject line is excluded from chunks because real inputs have none.
* The validation set is used for every choice (model selection, $\tau$). The test set is used **once**, for the final numbers.
* Metrics: accuracy, per-class precision/recall/F1, **macro-F1** (the unweighted mean of class F1s, so the small Medical and *Other*
  classes count as much as Politics), and confusion matrices.

### References
* Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*, 3, 993–1022.
* Hendrycks, D., & Gimpel, K. (2017). A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks. *ICLR*.
* Hoffman, M., Bach, F., & Blei, D. (2010). Online Learning for Latent Dirichlet Allocation. *NeurIPS*.
* Lang, K. (1995). NewsWeeder: Learning to Filter Netnews. *ICML*.
* Lee, D. D., & Seung, H. S. (1999). Learning the parts of objects by non-negative matrix factorization. *Nature*, 401, 788–791.
* Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
* Platt, J. (1999). Probabilistic outputs for support vector machines. *Advances in Large Margin Classifiers*.
* Röder, M., Both, A., & Hinneburg, A. (2015). Exploring the Space of Topic Coherence Measures. *WSDM*.
* Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5).
* Sievert, C., & Shirley, K. (2014). LDAvis: A method for visualizing and interpreting topics. *ACL Workshop on Interactive Language Learning*.